# La diversification résiste-t-elle aux baisses de marché ?

## Collecte des données

On utilise Yahoo Finance pour les cours de six ETF et le BLS pour
l’indice des prix américain. On commence par télécharger les sources
et vérifier les fichiers obtenus.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from fonctions.donnees import SUPPORTS

dossier_collecte = Path("data/raw")

On utilise les fichiers sauvegardés dans data/raw pour retrouver les mêmes
données à chaque exécution. La fonction telecharger_sources permet de
refaire séparément la collecte dans un nouveau dossier vide.

In [ ]:
fichiers_attendus = [
    *[f"{code}.csv" for code in SUPPORTS],
    "CPI_2004_2013.json",
    "CPI_2014_2023.json",
    "CPI_2024_2025.json",
    "collecte.json",
]


fichiers_absents = [
    nom for nom in fichiers_attendus
    if not (dossier_collecte / nom).is_file()
]

if fichiers_absents:
    raise FileNotFoundError(f"Collecte incomplète : {fichiers_absents}")

print("Les dix fichiers de collecte sont présents")

Les dix fichiers de collecte sont présents


On regarde ensuite les dates disponibles et les valeurs manquantes dans
les cours ajustés. On affiche aussi quelques lignes de SPY pour voir
comment les données sont organisées.

In [3]:
diagnostic = []

for code in SUPPORTS:
    cours = pd.read_csv(
        dossier_collecte / f"{code}.csv",
        parse_dates=["Date"],
    )

    diagnostic.append({
        "ETF": code,
        "première date": cours["Date"].min(),
        "dernière date": cours["Date"].max(),
        "nombre de lignes": len(cours),
        "cours ajustés manquants": cours["Adj Close"].isna().sum(),
    })

display(pd.DataFrame(diagnostic).set_index("ETF"))

spy = pd.read_csv(dossier_collecte / "SPY.csv")
display(spy.head())

,première date,dernière date,nombre de lignes,cours ajustés manquants
ETF,,,,
SPY,2005-12-30,2025-12-31,5032,0
EFA,2005-12-30,2025-12-31,5032,0
EEM,2005-12-30,2025-12-31,5032,0
IEF,2005-12-30,2025-12-31,5032,0
GLD,2005-12-30,2025-12-31,5032,0
SHY,2005-12-30,2025-12-31,5032,0


,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains
0,2005-12-30,124.800003,125.059998,124.360001,124.510002,85.285370,44645600,0.0,0.0,0.0
1,2006-01-03,125.190002,127.000000,124.389999,126.699997,86.785423,73256700,0.0,0.0,0.0
2,2006-01-04,126.860001,127.489998,126.699997,127.300003,87.196388,51899600,0.0,0.0,0.0
3,2006-01-05,127.150002,127.589996,126.879997,127.379997,87.251190,47307500,0.0,0.0,0.0
4,2006-01-06,128.020004,128.580002,127.360001,128.440002,87.977264,62885900,0.0,0.0,0.0


## Préparation des cours

On rassemble les cours ajustés sur les dates disponibles pour les six ETF.
On vérifie les valeurs manquantes et la période commune conservée.
Les données sont lues depuis la collecte sauvegardée dans data/raw.

In [4]:
from fonctions.donnees import preparer_prix

prix, qualite, comparaison = preparer_prix()

display(qualite)

print("Dimensions :", prix.shape)
print("Première date :", prix.index.min())
print("Dernière date :", prix.index.max())

display(prix.head())

,exposition,dates reçues,prix manquants
SPY,"actions américaines, S&P 500",5032,0
EFA,actions développées hors États-Unis et Canada,5032,0
EEM,actions émergentes,5032,0
IEF,obligations du Trésor américain à 7–10 ans,5032,0
GLD,or physique,5032,0
SHY,obligations du Trésor américain à 1–3 ans,5032,0


Dimensions : (5032, 6)
Première date : 2005-12-30 00:00:00
Dernière date : 2025-12-31 00:00:00


,SPY,EFA,EEM,IEF,GLD,SHY
Date,,,,,,
2005-12-30,85.285370,32.531055,19.416000,48.261982,51.580002,54.096439
2006-01-03,86.785423,33.620354,20.142040,48.256290,53.119999,54.170635
2006-01-04,87.196388,33.899513,20.324648,48.342514,53.299999,54.224575
2006-01-05,87.251190,33.729828,20.428049,48.354057,52.340000,54.177361
2006-01-06,87.977264,34.184162,20.854872,48.290752,53.720001,54.190914


On compare ensuite la variation des cours ajustés à celle des cours de
clôture. Cela permet de voir l’importance des distributions prises en
compte dans les ajustements.

In [5]:
display(comparaison.round(2))

,rendement ajusté (%),sans distributions (%)
SPY,693.38,447.68
EFA,190.60,61.59
EEM,180.32,85.98
IEF,92.73,14.60
GLD,668.34,668.34
SHY,48.63,3.25


## Préparation de l’inflation

On utilise l’indice mensuel des prix à la consommation américain fourni
par le BLS. On calcule sa variation sur douze mois pour décrire le contexte
d’inflation. Les mois absents restent manquants.

In [6]:
from fonctions.donnees import lire_inflation

inflation = lire_inflation()

display(inflation.head().to_frame())
display(inflation.tail().to_frame())

print("Nombre de mois :", len(inflation))
print("Mois renseignés :", inflation.notna().sum())

display(
    inflation[inflation.isna()]
    .to_frame()
    .rename_axis("mois manquant")
)

,inflation_pct
2006-01,4.018789
2006-02,3.638254
2006-03,3.417918
2006-04,3.613836
2006-05,3.977273


,inflation_pct
2025-08,2.938592
2025-09,3.022572
2025-10,NaN
2025-11,2.696444
2025-12,2.653304


Nombre de mois : 240
Mois renseignés : 239


,inflation_pct
mois manquant,
2025-10,NaN
